# Processing the data (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [9]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install --upgrade pip
# !pip install torch
# !pip install transformers[torch]
# !pip install scikit-learn

In [10]:
from datasets import load_dataset
import csv

# 1. Define the column names according to the LIAR dataset description
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

# 2. Load local files
raw_datasets = load_dataset(
    "csv", 
    data_files={
        "train": "/home/onyxia/work/Stat_App/Data/train.tsv", 
        "validation": "/home/onyxia/work/Stat_App/Data/valid.tsv", 
        "test": "/home/onyxia/work/Stat_App/Data/test.tsv"
    }, 
    delimiter="\t", 
    column_names=col_names,
    quoting=csv.QUOTE_NONE
)

# 3. Create a mapping for the labels (Text -> Integer)
label_mapping = {
    'pants-fire': 0, 
    'false': 1, 
    'barely-true': 2, 
    'half-true': 3, 
    'mostly-true': 4, 
    'true': 5
}

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

raw_datasets = raw_datasets.map(map_labels)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 10269
    })
    validation: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1283
    })
})


In [11]:
from transformers import AutoTokenizer, DataCollatorWithPadding

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["statement"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# cols_to_keep = ["input_ids", "attention_mask", "label"]
# tokenized_datasets = tokenized_datasets.remove_columns(
#     [c for c in tokenized_datasets["train"].column_names if c not in cols_to_keep]
# )

# Formatage pour PyTorch
#tokenized_datasets.set_format("torch")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 1284/1284 [00:00<00:00, 20980.10 examples/s]


# FINE TUNING

In [13]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

# 1. Charger la métrique "accuracy" (Précision)
# On utilise 'accuracy' car c'est une classification standard
metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    # On prend l'index de la plus haute probabilité (argmax) pour trouver la classe prédite (0 à 5)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. Configurer les hyperparamètres d'entraînement
training_args = TrainingArguments(
    output_dir="liar-bert-finetuned", # Dossier où le modèle sera sauvegardé
    eval_strategy="epoch",            # Évaluer le modèle à la fin de chaque époque
    save_strategy="epoch",            # Sauvegarder le checkpoint à la fin de chaque époque
    learning_rate=2e-5,               # Vitesse d'apprentissage recommandée pour BERT
    per_device_train_batch_size=16,   # Taille des lots (baissez à 8 si erreur de mémoire GPU)
    per_device_eval_batch_size=16,
    num_train_epochs=1,               # Nombre de fois que le modèle voit tout le dataset (3)
    weight_decay=0.01,
    load_best_model_at_end=True,      # À la fin, garder la meilleure version (pas forcément la dernière)
)

# 3. Charger le modèle vierge (Pre-trained) avec 6 labels
checkpoint = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=6)

# 4. Initialiser le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,             # Passe le tokenizer pour qu'il gère le padding final si besoin
    data_collator=data_collator,     # Votre collator défini précédemment
    compute_metrics=compute_metrics, # La fonction définie plus haut
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2520/3062619623.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
print("--- Évaluation finale sur le jeu de TEST ---")
# Le Trainer a une méthode dédiée pour lancer le calcul des métriques sur un dataset donné
test_results = trainer.evaluate(tokenized_datasets["test"])
print(test_results)

--- Évaluation finale sur le jeu de TEST ---


{'eval_loss': 1.6800254583358765, 'eval_accuracy': 0.25954793452844893, 'eval_runtime': 3.3484, 'eval_samples_per_second': 383.17, 'eval_steps_per_second': 24.191, 'epoch': 1.0}


In [ ]:
# Sauvegarde le modèle final et le tokenizer dans le dossier
trainer.save_model("liar-bert-finetuned-final")
tokenizer.save_pretrained("liar-bert-finetuned-final")
print("Modèle final sauvegardé avec succès.")

Modèle final sauvegardé avec succès.


In [ ]:
import torch

# Fonction pour tester une phrase
def predict_fake_news(text):
    # 1. Préparer le texte
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # 2. Envoyer sur le même appareil que le modèle (GPU ou CPU)
    device = trainer.model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 3. Prédiction
    with torch.no_grad():
        outputs = trainer.model(**inputs)
    
    # 4. Convertir les logits en probabilités
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    
    # Retrouver le label texte grâce à votre dictionnaire inversé
    # On inverse votre dictionnaire label_mapping
    id2label = {v: k for k, v in label_mapping.items()}
    return id2label[predicted_class_id]

# Testons !
fake_sentence = "The earth is completely flat and supported by a giant turtle."
real_sentence = "The sky is blue and water is wet."

print(f"Phrase 1: {predict_fake_news(fake_sentence)}")
print(f"Phrase 2: {predict_fake_news(real_sentence)}")

Phrase 1: false
Phrase 2: false
